## Projet Maintenance Prédictive
### Étape 2 — Préparation des données & Feature Engineering

**Objectif :** Transformer les séries temporelles brutes (PS2 : 6000 pts, FS1 : 600 pts)  
en un tableau de features numériques exploitable par un modèle sklearn.

**Plan :**
1. Chargement des données brutes
2. Création de la cible binaire
3. Extraction des features statistiques par cycle
4. Vérification du résultat
5. Split train / test (contrainte : 2000 premiers cycles = train)
6. Sauvegarde des données préparées


### 1. Imports

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

print("Imports OK")


Imports OK


### 2. Chargement des données brutes

In [3]:
# ── Adapter ce chemin si besoin ──
DATA_PATH = os.path.join("..", "data", "raw") + os.sep

# Vérification des fichiers
for fname in ["PS2.txt", "FS1.txt", "profile.txt"]:
    full = os.path.join(DATA_PATH, fname)
    status = "OK" if os.path.exists(full) else "INTROUVABLE"
    print(f"{fname:15s} → {status}")


PS2.txt         → OK
FS1.txt         → OK
profile.txt     → OK


In [4]:
ps2 = pd.read_csv(DATA_PATH + "PS2.txt", sep='\t', header=None)
fs1 = pd.read_csv(DATA_PATH + "FS1.txt", sep='\t', header=None)
profile = pd.read_csv(
    DATA_PATH + "profile.txt", sep='\t', header=None,
    names=['cooler', 'valve', 'pump', 'accumulator', 'stable']
)

print(f"PS2     : {ps2.shape}  → {ps2.shape[0]} cycles x {ps2.shape[1]} timesteps")
print(f"FS1     : {fs1.shape}  → {fs1.shape[0]} cycles x {fs1.shape[1]} timesteps")
print(f"Profile : {profile.shape}")


PS2     : (2205, 6000)  → 2205 cycles x 6000 timesteps
FS1     : (2205, 600)  → 2205 cycles x 600 timesteps
Profile : (2205, 5)


### 3. Création de la variable cible binaire

On transforme `valve condition` (4 classes : 73, 80, 90, 100) en cible binaire :
- **1** → valve = 100% (optimal)
- **0** → valve ∈ {73, 80, 90} (non optimal)


In [5]:
# Cible binaire
profile['valve_optimal'] = (profile['valve'] == 100).astype(int)

print("Distribution de la cible binaire :")
counts = profile['valve_optimal'].value_counts().sort_index()
for label, val in counts.items():
    name = "Optimal (100%)" if label == 1 else "Non optimal   "
    print(f"  Classe {label} — {name} : {val} cycles ({val/len(profile)*100:.1f}%)")


Distribution de la cible binaire :
  Classe 0 — Non optimal    : 1080 cycles (49.0%)
  Classe 1 — Optimal (100%) : 1125 cycles (51.0%)


### 4. Extraction des features statistiques

### Pourquoi des statistiques ?

Chaque cycle contient 6000 valeurs (PS2) ou 600 valeurs (FS1).  
On ne peut pas donner les 6600 valeurs brutes à un modèle sklearn directement —  
c'est trop large et redondant.

On **résume** chaque cycle avec des **statistiques descriptives** :

| Feature | Description |
|---------|-------------|
| `mean` | Moyenne → niveau général du signal |
| `std` | Écart-type → variabilité |
| `min` | Valeur minimale |
| `max` | Valeur maximale |
| `median` | Médiane (robuste aux outliers) |
| `q25` | 1er quartile |
| `q75` | 3ème quartile |
| `iqr` | Interquartile range (q75 - q25) |
| `skew` | Asymétrie de la distribution |
| `kurtosis` | Aplatissement de la distribution |
| `rms` | Root Mean Square (énergie du signal) |
| `peak_to_peak` | Amplitude max - min |

→ **12 features × 2 signaux = 24 features** par cycle


In [6]:
def extract_features(signal_df, prefix):
    """
    Extrait 12 features statistiques pour chaque cycle (ligne) d'un signal.
    
    Args:
        signal_df : DataFrame (n_cycles x n_timesteps)
        prefix    : str — préfixe du nom de colonne (ex: 'ps2', 'fs1')
    
    Returns:
        DataFrame (n_cycles x 12) avec les features nommées
    """
    arr = signal_df.values  # numpy array (n_cycles x n_timesteps)
    
    features = {
        f'{prefix}_mean'        : arr.mean(axis=1),
        f'{prefix}_std'         : arr.std(axis=1),
        f'{prefix}_min'         : arr.min(axis=1),
        f'{prefix}_max'         : arr.max(axis=1),
        f'{prefix}_median'      : np.median(arr, axis=1),
        f'{prefix}_q25'         : np.percentile(arr, 25, axis=1),
        f'{prefix}_q75'         : np.percentile(arr, 75, axis=1),
        f'{prefix}_iqr'         : np.percentile(arr, 75, axis=1) - np.percentile(arr, 25, axis=1),
        f'{prefix}_skew'        : stats.skew(arr, axis=1),
        f'{prefix}_kurtosis'    : stats.kurtosis(arr, axis=1),
        f'{prefix}_rms'         : np.sqrt((arr ** 2).mean(axis=1)),
        f'{prefix}_peak_to_peak': arr.max(axis=1) - arr.min(axis=1),
    }
    
    return pd.DataFrame(features)


print("Extraction features PS2...")
features_ps2 = extract_features(ps2, prefix='ps2')
print(f"  → {features_ps2.shape}  (12 features pour {features_ps2.shape[0]} cycles)")

print("Extraction features FS1...")
features_fs1 = extract_features(fs1, prefix='fs1')
print(f"  → {features_fs1.shape}  (12 features pour {features_fs1.shape[0]} cycles)")


Extraction features PS2...
  → (2205, 12)  (12 features pour 2205 cycles)
Extraction features FS1...
  → (2205, 12)  (12 features pour 2205 cycles)


### 5. Assemblage du dataset final

In [7]:
# Concaténation des features + stable flag + cible
X = pd.concat([
    features_ps2.reset_index(drop=True),
    features_fs1.reset_index(drop=True),
    profile[['stable']].reset_index(drop=True)   # stable flag comme feature
], axis=1)

y = profile['valve_optimal'].reset_index(drop=True)

print("=== Dataset final ===")
print(f"X shape : {X.shape}  ({X.shape[1]} features x {X.shape[0]} cycles)")
print(f"y shape : {y.shape}")
print()
print("Colonnes :")
print(X.columns.tolist())


=== Dataset final ===
X shape : (2205, 25)  (25 features x 2205 cycles)
y shape : (2205,)

Colonnes :
['ps2_mean', 'ps2_std', 'ps2_min', 'ps2_max', 'ps2_median', 'ps2_q25', 'ps2_q75', 'ps2_iqr', 'ps2_skew', 'ps2_kurtosis', 'ps2_rms', 'ps2_peak_to_peak', 'fs1_mean', 'fs1_std', 'fs1_min', 'fs1_max', 'fs1_median', 'fs1_q25', 'fs1_q75', 'fs1_iqr', 'fs1_skew', 'fs1_kurtosis', 'fs1_rms', 'fs1_peak_to_peak', 'stable']


In [8]:
# Aperçu des 5 premières lignes
print("Aperçu X (5 premières lignes) :")
X.head()


Aperçu X (5 premières lignes) :


,ps2_mean,ps2_std,ps2_min,ps2_max,ps2_median,ps2_q25,ps2_q75,ps2_iqr,ps2_skew,ps2_kurtosis,...,fs1_max,fs1_median,fs1_q25,fs1_q75,fs1_iqr,fs1_skew,fs1_kurtosis,fs1_rms,fs1_peak_to_peak,stable
0,109.466914,47.110581,0.0,156.99,129.365,120.1900,130.54,10.3500,-1.837853,1.508854,...,18.710,7.836,7.68775,7.93400,0.24625,-1.399797,1.661736,7.355220,18.710,1
1,109.354890,47.041690,0.0,157.56,129.385,120.1300,130.38,10.2500,-1.838351,1.511086,...,18.712,7.853,7.72400,7.95225,0.22825,-1.446472,1.637921,7.356488,18.712,1
2,109.158845,46.988144,0.0,156.97,129.325,120.0400,130.13,10.0900,-1.837545,1.508480,...,18.698,7.847,7.69875,7.95425,0.25550,-1.431863,1.624573,7.362682,18.698,1
3,109.064807,46.968307,0.0,156.44,128.865,119.9600,130.02,10.0600,-1.837120,1.507613,...,18.896,7.843,7.71100,7.95425,0.24325,-1.406670,1.655717,7.366970,18.896,1
4,108.931434,46.871040,0.0,158.13,129.000,119.8375,129.82,9.9825,-1.837496,1.509724,...,18.876,7.831,7.69025,7.94325,0.25300,-1.422662,1.619443,7.335840,18.876,1


In [9]:
# Statistiques descriptives du dataset
print("Statistiques descriptives :")
X.describe().round(4)


Statistiques descriptives :


,ps2_mean,ps2_std,ps2_min,ps2_max,ps2_median,ps2_q25,ps2_q75,ps2_iqr,ps2_skew,ps2_kurtosis,...,fs1_max,fs1_median,fs1_q25,fs1_q75,fs1_iqr,fs1_skew,fs1_kurtosis,fs1_rms,fs1_peak_to_peak,stable
count,2205.0000,2205.0000,2205.0,2205.0000,2205.0000,2205.0000,2205.0000,2205.0000,2205.0000,2205.0000,...,2205.0000,2205.0000,2205.0000,2205.0000,2205.0000,2205.0000,2205.0000,2205.0000,2205.0000,2205.0000
mean,109.3799,47.7322,0.0,166.5205,128.6777,118.2318,132.5068,14.2750,-1.7754,1.3600,...,20.1302,6.9903,6.8578,7.6939,0.8361,-1.0083,1.9575,6.9409,20.1302,0.3429
std,4.9866,3.2717,0.0,0.9780,3.4695,1.7921,11.4683,12.2795,0.1401,0.2927,...,0.4519,1.9975,2.0140,0.4927,1.8763,0.7812,0.4703,0.7583,0.4519,0.4748
min,104.4063,45.2032,0.0,155.0400,124.4450,112.8400,125.0400,9.1225,-1.8423,0.4553,...,18.6980,1.0970,1.0460,1.1540,0.0500,-1.4961,-0.3508,3.3489,18.6980,0.0000
25%,106.9624,46.2058,0.0,166.1800,127.2150,116.6100,127.7300,9.9100,-1.8355,1.3970,...,19.8810,7.4810,7.3740,7.5860,0.1790,-1.3437,1.8179,7.0250,19.8810,0.0000
50%,107.7302,46.8021,0.0,166.6500,127.9650,118.0900,128.5200,10.0300,-1.8299,1.4762,...,20.3630,7.6970,7.6130,7.7540,0.2140,-1.2584,1.9455,7.2263,20.3630,0.0000
75%,109.4216,47.2036,0.0,167.1000,129.9500,120.2600,130.5100,10.1200,-1.8063,1.5063,...,20.4790,7.8005,7.6838,7.8880,0.2323,-1.1777,2.1287,7.3083,20.4790,1.0000
max,131.5891,59.5483,0.0,167.7700,165.4000,120.9600,165.5600,51.8000,-1.3176,1.5222,...,20.4790,7.8530,7.7278,7.9542,6.6592,2.5025,7.6604,7.3777,20.4790,1.0000


### 6. Vérifications qualité

In [10]:
print("=== Vérification NaN ===")
nan_count = X.isna().sum()
if nan_count.sum() == 0:
    print("  Aucune valeur manquante")
else:
    print(nan_count[nan_count > 0])

print()
print("=== Vérification Inf ===")
inf_count = np.isinf(X.values).sum()
if inf_count == 0:
    print("  Aucune valeur infinie")
else:
    print(f"  {inf_count} valeurs infinies détectées !")

print()
print("=== Distribution de la cible ===")
print(y.value_counts().sort_index())


=== Vérification NaN ===
  Aucune valeur manquante

=== Vérification Inf ===
  Aucune valeur infinie

=== Distribution de la cible ===
valve_optimal
0    1080
1    1125
Name: count, dtype: int64


### 7. Split Train / Test

**Contrainte du projet :**
- Les **2000 premiers cycles** → jeu d'entraînement
- Les **cycles restants** (205) → jeu de test final

On ne mélange **pas** les données (`shuffle=False`).  
C'est un split temporel : on entraîne sur le passé, on teste sur le futur.


In [11]:
TRAIN_SIZE = 2000

X_train = X.iloc[:TRAIN_SIZE].reset_index(drop=True)
X_test  = X.iloc[TRAIN_SIZE:].reset_index(drop=True)
y_train = y.iloc[:TRAIN_SIZE].reset_index(drop=True)
y_test  = y.iloc[TRAIN_SIZE:].reset_index(drop=True)

print("=== Split Train / Test ===")
print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")
print()
print("Distribution y_train :")
c = y_train.value_counts().sort_index()
for label, val in c.items():
    print(f"  Classe {label} : {val} ({val/len(y_train)*100:.1f}%)")
print()
print("Distribution y_test :")
c = y_test.value_counts().sort_index()
for label, val in c.items():
    print(f"  Classe {label} : {val} ({val/len(y_test)*100:.1f}%)")


=== Split Train / Test ===
X_train : (2000, 25)  |  y_train : (2000,)
X_test  : (205, 25)   |  y_test  : (205,)

Distribution y_train :
  Classe 0 : 948 (47.4%)
  Classe 1 : 1052 (52.6%)

Distribution y_test :
  Classe 0 : 132 (64.4%)
  Classe 1 : 73 (35.6%)


### 8. Sauvegarde des données préparées

In [12]:
PROCESSED_PATH = os.path.join("..", "data", "processed") + os.sep
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Sauvegarde
X_train.to_csv(PROCESSED_PATH + "X_train.csv", index=False)
X_test.to_csv( PROCESSED_PATH + "X_test.csv",  index=False)
y_train.to_csv(PROCESSED_PATH + "y_train.csv", index=False)
y_test.to_csv( PROCESSED_PATH + "y_test.csv",  index=False)

print("Fichiers sauvegardés dans data/processed/ :")
for fname in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]:
    fpath = PROCESSED_PATH + fname
    size  = os.path.getsize(fpath) / 1024
    print(f"  {fname:20s} → {size:.1f} KB")


Fichiers sauvegardés dans data/processed/ :
  X_train.csv          → 599.7 KB
  X_test.csv           → 62.0 KB
  y_train.csv          → 5.9 KB
  y_test.csv           → 0.6 KB


### 9. Résumé

| Étape | Résultat |
|-------|----------|
| Signaux bruts | PS2 (6000 pts/cycle) + FS1 (600 pts/cycle) |
| Features extraites | 12 stats × 2 signaux + 1 stable flag = **25 features** |
| Cycles total | 2205 |
| **X_train** | 2000 cycles × 25 features |
| **X_test** | 205 cycles × 25 features |
| NaN / Inf | Aucun |
| Classes train | Équilibrées |

### Fichiers générés
```
data/processed/
├── X_train.csv   ← features des 2000 cycles d'entraînement
├── X_test.csv    ← features des 205 cycles de test
├── y_train.csv   ← cible train (0/1)
└── y_test.csv    ← cible test  (0/1)
```

### Prochaine étape
→ **03_model_training.ipynb** : entraînement du modèle sklearn sur X_train/y_train
